### 0. Preliminaries
Import necessary packages

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

### 1. Model Setup
Define constant parameters, user functions, and simulation individuals

In [28]:
# Define constant parameters
county_size = 20 # miles
taxi_speed = 20 # miles per hour
base_fare = 3.00 # pounds
per_mile_fare = 2.00 # pounds per mile
petrol_cost = 0.2 # pounds per mile

In [29]:
def distance(loc1, loc2):
    return np.sqrt((loc1[0] - loc2[0])**2 + (loc1[1] - loc2[1])**2)

def generate_location():
    return (random.uniform(0, county_size), random.uniform(0, county_size))

def travel_time(d_OD):
    mu = d_OD / taxi_speed
    return random.uniform(0.8*mu, 1.2*mu) 

### 2. Simulation

In [30]:
TNOW = 0
# insert performance metrics here (alice)
Termination = 100 #hrs
EventTypes = ['DriverArrival', 'RiderArrival', 'PickupStart', 
              'Pickup', 'Dropoff', 'RiderAbandonment', 'DriverOffline', 'Termination']
EventCalendar = []

def schedule_event(time, event_type, data=None):
    '''Event scheduler that keeps track of times, event type, and user data (optional)'''
    EventCalendar.append((time,event_type,data))

drivers =[]
idle_drivers=[]
riders = []
waiting_riders=[]

driver_id_counter = 0
rider_id_counter = 0

completed_rides = 0
abandoned_rides = 0

schedule_event(random.expovariate(3), 1)   # 1 = Driver Arrival
schedule_event(random.expovariate(30), 2)    # 2 = Rider Arrival
schedule_event(Termination, 8)                      # 8 = Termination

In [ ]:
while TNOW < Termination:
    TNEXT, TypeNext, Data = min(EventCalendar, key=lambda x: x[0])
    EventCalendar.remove((TNEXT, TypeNext, Data))

    ## Update Performance Stats Here (Alice)

    TNOW = TNEXT

    # DriverArrival
    if TypeNext == 1:
        driver_id_counter += 1

        # Generate driver location and offline time (initialize other attributes)
        driver = {
            "id": driver_id_counter,
            "location": generate_location(),
            'offline_time': TNOW + random.uniform(5,8),
            'idle': True,
            'active': True,
            'earnings': 0
        }
        drivers.append(driver)
        
        # Schedule offline time
        schedule_event(driver['offline_time'], event_type=7, data = driver)

        # Match driver to waiting rider
        if waiting_riders:
            # match closest waiting rider
            rider = min(waiting_riders, key = lambda r:distance(driver['location'], r['origin']))
            waiting_riders.remove(rider)
            
            # update statuses
            rider['status'] = 'matched'
            driver['idle'] = False
            
            # schedule pickup start
            schedule_event(TNOW, event_type=3, data = (driver, rider)) 
        else:
            idle_drivers.append(driver)
        
        # Schedule next driver arrival
        schedule_event(TNOW + random.expovariate(3), event_type=1)

    # RiderArrival
    elif TypeNext == 2: 
        rider_id_counter += 1

        # Generate rider location and patience time (initialize other attributes)
        rider = {
            "id": rider_id_counter,
            "origin": generate_location(),
            'destination': generate_location(),
            'patience_time': random.expovariate(5),
            'status': 'waiting',
        }
        
        # Match driver to idle driver
        if idle_drivers:
            # match to closest driver
            driver = min(idle_drivers, key = lambda d:distance(d['location'], rider['origin']))
            idle_drivers.remove(driver)

            # update statuses
            rider['status'] = 'matched'
            driver['idle'] = False

            # schedule PickupStart
            schedule_event(TNOW, event_type=3, data = (driver,rider))
        else:
            waiting_riders.append(rider)

            #schedule patience deadline for waiting rider
            schedule_event(TNOW+rider['patience_time'], event_type=6, data = rider)
        
        # schedule next rider arrival
        schedule_event(TNOW + random.expovariate(30), event_type=2)

    # PickupStart
    elif TypeNext == 3:
        driver,rider = Data

        if not driver['active']:
            continue
        
        # Calculate actual travel time 
        d1 = distance(driver['location'],rider['origin'])
        t1 = travel_time(d1)

        # Schedule Pickup event
        schedule_event(TNOW + t1, event_type=4, data = (driver,rider,d1))
    
    # Pickup
    elif TypeNext == 4:
        driver, rider, d1 = Data
        
        # Calculate actual travel time
        d2 = distance(rider['origin'], rider['destination'])
        t2 = travel_time(d2)

        # Schedule Dropoff
        schedule_event(TNOW + t2, event_type=5, data = (driver,rider,d1,d2))

    # Dropoff
    elif TypeNext == 5:
        driver, rider, d1, d2 = Data

        # complete the ride and update rider status
        completed_rides += 1
        rider["status"] = "dropped-off"

        # calculate driver earnings
        rider_fare = base_fare + per_mile_fare * d2
        driver_cost = petrol_cost*(d1+d2)
        driver['earnings'] =  rider_fare - petrol_cost
        
        # Update driver information
        driver['location'] = rider['destination']
        driver['idle'] = True

        # Check if driver is meant to be offline now
        if TNOW >= driver['offline_time']:
            driver['active'] = False
        else:
            if waiting_riders:
                next_rider = min(waiting_riders,
                                 key=lambda r: distance(driver['location'], r['origin']))
                waiting_riders.remove(next_rider)
                next_rider['status'] = 'matched'
                driver['idle'] = False
                # schedule next pickup if driver gets matched to a new rider
                schedule_event(TNOW, event_type=3, data = (driver, next_rider))
            else:
                idle_drivers.append(driver)
    
    # RiderAbandonment
    elif TypeNext == 6:
        rider = Data

        # remove abandonded riders and update status
        if rider['status'] == 'waiting':
            waiting_riders.remove(rider)
            rider["status"] = "abandoned"
            abandoned_rides += 1
    
    # DriverOffline
    elif TypeNext == 7:
        driver = Data
        
        # remove offline drivers and update status
        if driver["active"] and driver["idle"]:
            driver["active"] = False
            if driver in idle_drivers:
                idle_drivers.remove(driver)

    else:
        ## final performance metric stats calc here (alice)
        break








In [34]:
EventCalendar

[(100.63430281520795,
  7,
  {'id': 303,
   'location': (3.1419618698494856, 1.2618771432875953),
   'offline_time': 100.63430281520795,
   'idle': False,
   'active': True,
   'earnings': np.float64(40.18384352745056)}),
 (101.6411571470475,
  7,
  {'id': 306,
   'location': (18.217134853256308, 10.104225254719898),
   'offline_time': 101.6411571470475,
   'idle': False,
   'active': True,
   'earnings': np.float64(17.51759618906185)}),
 (101.83544098440377,
  7,
  {'id': 307,
   'location': (2.908098368664993, 19.591887585736224),
   'offline_time': 101.83544098440377,
   'idle': False,
   'active': True,
   'earnings': np.float64(36.1798698106338)}),
 (103.00932550425539,
  7,
  {'id': 308,
   'location': (12.64375265702762, 10.27908759391984),
   'offline_time': 103.00932550425539,
   'idle': False,
   'active': True,
   'earnings': np.float64(9.946335166295285)}),
 (100.91394755398068,
  7,
  {'id': 309,
   'location': (11.665915584785719, 15.746514727465543),
   'offline_time': 1

### 2. Analyze Simulation Results

### 3. Manually Simulate with Given Data

The `riders.xlsx` and `drivers.xlsx` data sets are provided from the project assignment page. 

In [6]:
riders_df = pd.read_excel('riders.xlsx')
drivers_df = pd.read_excel('drivers.xlsx')